# Enhanced Retrieval v2 - Aggressive Optimization

## Diagnosis: Why 22% instead of 45%+?

Possible issues:
1. Query preprocessing too aggressive (losing important terms)
2. Retrieved documents don't contain answers
3. Prompt not extractive enough
4. Answer cleaning too aggressive
5. Context window too small

## New Strategies:
1. **Less aggressive query preprocessing** - Keep more terms
2. **Wider retrieval net** - Retrieve more documents (k=20-30)
3. **Better prompts** - More explicit extraction instructions
4. **Longer contexts** - Give LLM more to work with
5. **Debug mode** - See what's being retrieved

In [ ]:
# Setup
import pandas as pd
import json
import re
import string
from collections import Counter
import numpy as np

from pyserini.search import SimpleSearcher
from pyserini.index.lucene import IndexReader
import transformers
import torch

# Load index
searcher = SimpleSearcher.from_prebuilt_index('wikipedia-kilt-doc')
index_reader = IndexReader.from_prebuilt_index('wikipedia-kilt-doc')
print("Index loaded:", index_reader.stats())

In [ ]:
# Load data
from google.colab import drive
drive.mount('/content/drive')

train_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/train.csv"
test_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/test.csv"

df_train = pd.read_csv(train_path, converters={"answers": json.loads})
df_test = pd.read_csv(test_path)

print(f"Train: {len(df_train)}, Test: {len(df_test)}")

In [ ]:
# Load LLM
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")

model_id = "meta-llama/Llama-3.2-1B-Instruct"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("LLM loaded")

## REVISED: Less Aggressive Query Processing

In [ ]:
def preprocess_query_v2(query):
    """
    LESS aggressive preprocessing - keep more information.
    Only remove truly useless patterns.
    """
    query = query.strip()
    
    # Just remove question mark - that's it!
    query = query.rstrip('?').strip()
    
    return query


def expand_query_v2(query):
    """
    Generate more diverse query variations.
    """
    queries = []
    
    # 1. Original query (just remove ?)
    queries.append(preprocess_query_v2(query))
    
    # 2. Lowercase version
    lower_q = query.lower().rstrip('?').strip()
    if lower_q != queries[0]:
        queries.append(lower_q)
    
    # 3. Extract just the subject (after question word)
    # "what is the capital of France" -> "capital of France"
    pattern = r'^(what|where|when|who|which|how)\s+(is|are|was|were|does|do|did)\s+(the\s+)?(.+)$'
    match = re.match(pattern, lower_q, re.IGNORECASE)
    if match:
        subject = match.group(4).strip()
        if subject and subject not in queries:
            queries.append(subject)
    
    # 4. Keep ONLY the nouns (named entities likely)
    # Remove very common words but keep important ones
    minimal_stops = {'what', 'where', 'when', 'who', 'which', 'how', 'is', 'are', 'was', 'were', 'do', 'does', 'did'}
    tokens = lower_q.split()
    important_tokens = [t for t in tokens if t not in minimal_stops and len(t) > 2]
    if len(important_tokens) >= 2:
        important_q = ' '.join(important_tokens)
        if important_q not in queries:
            queries.append(important_q)
    
    return queries


# Test
test_q = "what is the name of justin bieber brother?"
print(f"Original: {test_q}")
for i, q in enumerate(expand_query_v2(test_q), 1):
    print(f"  {i}. {q}")

## REVISED: Retrieve MORE Documents

In [ ]:
def retrieve_with_fusion_v2(query, k=20, mu=1000):
    """
    Retrieve MORE documents to increase chance of finding answer.
    """
    searcher.set_qld(mu=mu)
    
    query_variations = expand_query_v2(query)
    
    doc_info = {}
    
    for q_var in query_variations:
        hits = searcher.search(q_var, k)
        
        for hit in hits:
            doc_id = hit.docid
            
            if doc_id not in doc_info:
                doc = searcher.doc(doc_id)
                data = json.loads(doc.raw())
                
                doc_info[doc_id] = {
                    'content': data.get('contents', ''),
                    'title': data.get('title', ''),
                    'score': hit.score,
                    'count': 1
                }
            else:
                # Boost heavily if appears in multiple queries
                doc_info[doc_id]['score'] += hit.score * 0.8  # Increased from 0.5
                doc_info[doc_id]['count'] += 1
    
    # Sort by score
    sorted_docs = sorted(doc_info.values(), key=lambda x: x['score'], reverse=True)[:k]
    
    return sorted_docs

## REVISED: Better Prompts - More Extractive

In [ ]:
def create_extractive_prompt(question, docs, max_doc_length=800):
    """
    Create a prompt that FORCES extraction, not generation.
    """
    # Format documents with more context
    doc_texts = []
    for i, doc in enumerate(docs[:15]):  # Use top 15 docs
        content = doc['content'].replace('\n', ' ')[:max_doc_length]
        title = doc['title']
        doc_texts.append(f"[{i+1}] {title}\n{content}")
    
    context = "\n\n".join(doc_texts)
    
    # Very explicit extraction prompt
    system = (
        "You are a precise answer extraction system. "
        "Your ONLY job is to find and copy the exact answer from the documents.\n\n"
        "RULES:\n"
        "1. Find the answer in the documents below\n"
        "2. Copy ONLY the specific name, place, date, or fact that answers the question\n"
        "3. Your answer must be 1-5 words maximum\n"
        "4. Do NOT add any explanation, context, or extra words\n"
        "5. If you cannot find the answer, respond with 'unknown'"
    )
    
    user = (
        f"DOCUMENTS:\n{context}\n\n"
        f"QUESTION: {question}\n\n"
        f"Extract the answer (1-5 words only):"
    )
    
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

## REVISED: Gentler Answer Cleaning

In [ ]:
def clean_answer_v2(answer):
    """
    Less aggressive cleaning - preserve more of the answer.
    """
    answer = answer.strip()
    
    # Only remove very obvious prefixes
    if answer.lower().startswith('the answer is '):
        answer = answer[14:].strip()
    elif answer.lower().startswith('answer: '):
        answer = answer[8:].strip()
    
    # Remove quotes
    answer = answer.strip('"\"''').strip()
    
    # Remove ONLY trailing period if answer is long
    if len(answer.split()) > 3 and answer.endswith('.'):
        answer = answer[:-1].strip()
    
    # Handle unknown
    if answer.lower() in ['unknown', 'i dont know', "i don't know"]:
        return 'unknown'
    
    return answer.strip()

## Main Pipeline with Debug Mode

In [ ]:
def answer_question_v2(question, k=20, mu=1000, temperature=0.1, debug=False):
    """
    Enhanced pipeline with debugging.
    """
    # Retrieve documents
    docs = retrieve_with_fusion_v2(question, k=k, mu=mu)
    
    if not docs:
        return 'unknown'
    
    if debug:
        print(f"\nQuestion: {question}")
        print(f"Retrieved {len(docs)} documents")
        print("\nTop 3 documents:")
        for i, doc in enumerate(docs[:3], 1):
            print(f"{i}. {doc['title']} (score: {doc['score']:.2f}, hits: {doc['count']})")
            print(f"   {doc['content'][:200]}...\n")
    
    # Create prompt
    messages = create_extractive_prompt(question, docs)
    
    # Generate
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=50,  # Shorter to force brevity
            eos_token_id=terminators,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
        
        raw = outputs[0]["generated_text"][-1].get('content', 'unknown')
        cleaned = clean_answer_v2(raw)
        
        if debug:
            print(f"Raw answer: '{raw}'")
            print(f"Cleaned: '{cleaned}'")
        
        return cleaned
    
    except Exception as e:
        if debug:
            print(f"Error: {e}")
        return 'unknown'

## Debug: Check What's Happening

In [ ]:
# Test with debug mode on a few examples
test_samples = df_train.sample(n=3, random_state=42)

for idx, row in test_samples.iterrows():
    print("=" * 80)
    answer = answer_question_v2(row['question'], debug=True)
    print(f"\nFinal Answer: '{answer}'")
    print(f"Ground Truth: {row['answers']}")
    print()

## Evaluation Functions

In [ ]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return int(pred_tokens == gt_tokens)
    if num_same == 0:
        return 0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def evaluate(predictions_dict, df_gold):
    f1_sum = 0.0
    count = 0
    
    for idx, row in df_gold.iterrows():
        qid = row['id']
        if qid not in predictions_dict:
            continue
        
        prediction = predictions_dict[qid]
        ground_truths = row['answers']
        
        f1 = max(f1_score(prediction, gt) for gt in ground_truths)
        f1_sum += f1
        count += 1
    
    return (100.0 * f1_sum / count) if count > 0 else 0.0

## REVISED Hyperparameter Tuning

In [ ]:
# Test on larger sample with new parameters
sample_size = 100
df_sample = df_train.sample(n=sample_size, random_state=42)

print(f"Testing on {sample_size} examples with REVISED strategy...\n")

# New parameter ranges focusing on wider retrieval
configs = [
    # More documents, lower temperature
    {'k': 20, 'mu': 1000, 'temp': 0.1}, #F1 Score: 29.66%
    {'k': 25, 'mu': 1000, 'temp': 0.1}, #F1 Score: 31.46%
    {'k': 30, 'mu': 1000, 'temp': 0.1}, #F1 Score: 32.12%
    
    # Try different mu with wider k
    {'k': 20, 'mu': 500, 'temp': 0.1}, #F1 Score: 27.71%
    {'k': 20, 'mu': 2000, 'temp': 0.1}, #F1 Score: 29.78%
    
    # Try slightly higher temp
    {'k': 20, 'mu': 1000, 'temp': 0.2}, #F1 Score: 29.86%
]

best_score = 0
best_config = None

for i, config in enumerate(configs):
    print(f"\nConfig {i+1}/{len(configs)}: {config}")
    
    predictions = {}
    for idx, row in df_sample.iterrows():
        predictions[row['id']] = answer_question_v2(
            row['question'],
            k=config['k'],
            mu=config['mu'],
            temperature=config['temp']
        )
    
    score = evaluate(predictions, df_sample)
    print(f"F1 Score: {score:.2f}%")
    
    if score > best_score:
        best_score = score
        best_config = config
        print("  ⭐ New best!")

print("\n" + "=" * 80)
print(f"REVISED Best: {best_config}")
print(f"REVISED Score: {best_score:.2f}%")
print("=" * 80)

## Generate Test Predictions with Best Config

In [ ]:
# Use best config
FINAL_K = 20  # Or from best_config
FINAL_MU = 1000
FINAL_TEMP = 0.1

print(f"Generating test predictions with k={FINAL_K}, mu={FINAL_MU}, temp={FINAL_TEMP}...\n")

test_predictions = {}

for idx, row in df_test.iterrows():
    answer = answer_question_v2(
        row['question'],
        k=FINAL_K,
        mu=FINAL_MU,
        temperature=FINAL_TEMP
    )
    test_predictions[row['id']] = answer
    
    if (idx + 1) % 100 == 0:
        print(f"  {idx + 1}/{len(df_test)}")
    
    if idx < 5:
        print(f"Q: {row['question']}")
        print(f"A: {answer}\n")

print("\n✓ Complete!")

## Save with Correct Format

In [ ]:
# Create submission
submission_df = pd.DataFrame([
    {'id': qid, 'prediction': answer}
    for qid, answer in test_predictions.items()
])

submission_df = submission_df.sort_values('id').reset_index(drop=True)

# CRITICAL: Format with ensure_ascii=False
submission_df['prediction'] = submission_df['prediction'].apply(
    lambda x: json.dumps([x], ensure_ascii=False)
)

# Save
output_path = "/content/drive/MyDrive/q-a-rag-assignment-3-reichman-uni/enhanced_v2_predictions.csv"
submission_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total predictions: {len(submission_df)}")
print("\nFirst 3:")
print(submission_df.head(3))